In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from pyproj import CRS, Transformer
import numpy as np

def get_unit_to_meter_conversion(unit_name):
    """
    Converte diferentes unidades para metros.
    Retorna o fator de conversão.
    """
    conversions = {
        'metre': 1.0,
        'meter': 1.0,
        'm': 1.0,
        'foot': 0.3048,
        'feet': 0.3048,
        'ft': 0.3048,
        'us survey foot': 0.3048006096012192,
        'foot_us': 0.3048006096012192,
        'link': 0.201168,
        'chain': 20.1168,
        'kilometre': 1000.0,
        'kilometer': 1000.0,
        'km': 1000.0,
    }
    return conversions.get(unit_name.lower(), None)


def calculate_raster_area_hectares(image_path, verbose=True):
    """
    Calcula a área em hectares de uma imagem raster (GeoTIFF).
    
    Esta função funciona com qualquer CRS, fazendo conversões automáticas quando necessário:
    - CRS projetados com unidades conhecidas (metros, pés, etc.) -> conversão direta
    - CRS geográficos (lat/lon) -> transforma para UTM apropriado
    - CRS com unidades desconhecidas -> transforma via geográfico para UTM
    
    Parameters:
    -----------
    image_path : str
        Caminho para o arquivo raster (GeoTIFF)
    verbose : bool, optional
        Se True, exibe informações detalhadas durante o cálculo (padrão: True)
    
    Returns:
    --------
    dict
        Dicionário com as seguintes chaves:
        - 'width_m': largura em metros
        - 'height_m': altura em metros
        - 'area_m2': área em metros quadrados
        - 'area_hectares': área em hectares
        - 'area_km2': área em quilômetros quadrados
        - 'crs': CRS original da imagem
        - 'dimensions': tupla (width_pixels, height_pixels)
    
    Raises:
    -------
    ValueError
        Se a imagem não possuir CRS definido ou se o CRS não puder ser processado
    FileNotFoundError
        Se o arquivo não for encontrado
    
    Examples:
    ---------
    >>> result = calculate_raster_area_hectares('minha_imagem.tif')
    >>> print(f"Área: {result['area_hectares']:.2f} hectares")
    """
    
    # Abrir o raster
    with rasterio.open(image_path) as src:
        # Informações básicas
        if verbose:
            print(f"Dimensões: {src.width} x {src.height} pixels")
            print(f"CRS original: {src.crs}")
            print(f"Bounds: {src.bounds}")
            print(f"Transform: {src.transform}")
        
        # Obter o CRS
        crs = src.crs
        
        if crs is None:
            raise ValueError("⚠️  Imagem não possui CRS definido! Não é possível calcular a área.")
        
        # Verificar o tipo de CRS
        crs_obj = CRS.from_string(str(crs))
        is_geographic = crs_obj.is_geographic
        is_projected = crs_obj.is_projected
        
        if verbose:
            print(f"\nTipo de CRS:")
            print(f"  - Geográfico (lat/lon): {is_geographic}")
            print(f"  - Projetado: {is_projected}")
        
        # Obter informações sobre as unidades
        if len(crs_obj.axis_info) >= 2:
            unit_x = crs_obj.axis_info[0].unit_name
            unit_y = crs_obj.axis_info[1].unit_name
            if verbose:
                print(f"  - Unidade X: {unit_x}")
                print(f"  - Unidade Y: {unit_y}")
        else:
            unit_x = unit_y = crs_obj.axis_info[0].unit_name if crs_obj.axis_info else 'unknown'
            if verbose:
                print(f"  - Unidade: {unit_x}")
        
        # Obter resolução do pixel
        pixel_width = abs(src.transform[0])
        pixel_height = abs(src.transform[4])
        
        if verbose:
            print(f"\nResolução do pixel: {pixel_width} x {pixel_height} ({unit_x})")
        
        # Estratégia de cálculo baseada no tipo de CRS
        if is_projected:
            # CRS projetado - verificar se está em metros ou outra unidade
            conversion_factor = get_unit_to_meter_conversion(unit_x)
            
            if conversion_factor is not None:
                # Podemos converter diretamente para metros
                if verbose:
                    print(f"\n✓ CRS projetado em {unit_x}. Convertendo para metros...")
                    print(f"  Fator de conversão: {conversion_factor}")
                
                width_m = src.width * pixel_width * conversion_factor
                height_m = src.height * pixel_height * conversion_factor
                area_m2 = width_m * height_m
                
            else:
                # Unidade desconhecida - usar transformação
                if verbose:
                    print(f"\n⚠️  Unidade '{unit_x}' não reconhecida. Usando transformação para UTM...")
                
                # Calcular centroide e transformar
                bounds = src.bounds
                center_x = (bounds.left + bounds.right) / 2
                center_y = (bounds.bottom + bounds.top) / 2
                
                # Tentar transformar para geográfico primeiro, depois para UTM
                try:
                    transformer_to_geo = Transformer.from_crs(crs, 'EPSG:4326', always_xy=True)
                    center_lon, center_lat = transformer_to_geo.transform(center_x, center_y)
                except:
                    raise ValueError("Não foi possível determinar coordenadas geográficas do raster.")
                
                # Determinar zona UTM
                utm_zone = int((center_lon + 180) / 6) + 1
                hemisphere = 'north' if center_lat >= 0 else 'south'
                
                if hemisphere == 'north':
                    dst_crs = CRS.from_string(f'EPSG:326{utm_zone:02d}')
                else:
                    dst_crs = CRS.from_string(f'EPSG:327{utm_zone:02d}')
                
                if verbose:
                    print(f"  CRS de destino (UTM): {dst_crs}")
                
                # Transformar bounds
                transformer = Transformer.from_crs(crs, dst_crs, always_xy=True)
                corners = [
                    (bounds.left, bounds.bottom),
                    (bounds.right, bounds.bottom),
                    (bounds.right, bounds.top),
                    (bounds.left, bounds.top)
                ]
                corners_transformed = [transformer.transform(x, y) for x, y in corners]
                
                width_m = abs(corners_transformed[1][0] - corners_transformed[0][0])
                height_m = abs(corners_transformed[2][1] - corners_transformed[1][1])
                area_m2 = width_m * height_m
                
        elif is_geographic:
            # CRS geográfico (lat/lon) - precisamos transformar para UTM
            if verbose:
                print(f"\n⚠️  CRS geográfico (lat/lon). Transformando para UTM...")
            
            bounds = src.bounds
            center_lon = (bounds.left + bounds.right) / 2
            center_lat = (bounds.bottom + bounds.top) / 2
            
            # Determinar zona UTM apropriada
            utm_zone = int((center_lon + 180) / 6) + 1
            hemisphere = 'north' if center_lat >= 0 else 'south'
            
            # Criar CRS UTM
            if hemisphere == 'north':
                dst_crs = CRS.from_string(f'EPSG:326{utm_zone:02d}')
            else:
                dst_crs = CRS.from_string(f'EPSG:327{utm_zone:02d}')
            
            if verbose:
                print(f"  Zona UTM: {utm_zone}{hemisphere[0].upper()}")
                print(f"  CRS de destino: {dst_crs}")
            
            # Transformar os bounds para o CRS métrico
            transformer = Transformer.from_crs(crs, dst_crs, always_xy=True)
            
            # Transformar os 4 cantos
            corners = [
                (bounds.left, bounds.bottom),
                (bounds.right, bounds.bottom),
                (bounds.right, bounds.top),
                (bounds.left, bounds.top)
            ]
            
            corners_transformed = [transformer.transform(x, y) for x, y in corners]
            
            # Calcular largura e altura em metros
            width_m = abs(corners_transformed[1][0] - corners_transformed[0][0])
            height_m = abs(corners_transformed[2][1] - corners_transformed[1][1])
            
            area_m2 = width_m * height_m
            
        else:
            raise ValueError("⚠️  Tipo de CRS não reconhecido! Não é possível calcular a área.")
        
        # Converter para hectares (1 hectare = 10,000 m²)
        area_hectares = area_m2 / 10000
        area_km2 = area_hectares / 100
        
        if verbose:
            print(f"\n{'='*50}")
            print(f"RESULTADOS:")
            print(f"{'='*50}")
            print(f"Largura: {width_m:.2f} metros")
            print(f"Altura: {height_m:.2f} metros")
            print(f"Área total: {area_m2:,.2f} m²")
            print(f"Área total: {area_hectares:,.2f} hectares")
            print(f"Área total: {area_km2:,.2f} km²")
            print(f"{'='*50}")
        
        return {
            'width_m': width_m,
            'height_m': height_m,
            'area_m2': area_m2,
            'area_hectares': area_hectares,
            'area_km2': area_km2,
            'crs': str(crs),
            'dimensions': (src.width, src.height)
        }


In [ ]:
def calculate_non_empty_area_hectares(image_path, verbose=True):
    """
    Calcula a área em hectares apenas dos pixels não vazios (não-zeros) de uma imagem raster.
    
    Esta função é agnóstica ao CRS/projeção e funciona com imagens RGB ou em escala de cinza.
    Pixels são considerados vazios se todos os seus canais forem zero.
    
    Parameters:
    -----------
    image_path : str
        Caminho para o arquivo raster (GeoTIFF)
    verbose : bool, optional
        Se True, exibe informações detalhadas durante o cálculo (padrão: True)
    
    Returns:
    --------
    dict
        Dicionário com as seguintes chaves:
        - 'total_pixels': número total de pixels na imagem
        - 'non_empty_pixels': número de pixels não vazios
        - 'empty_pixels': número de pixels vazios (todos canais = 0)
        - 'percent_non_empty': percentual de pixels não vazios
        - 'pixel_area_m2': área de um pixel em metros quadrados
        - 'non_empty_area_m2': área não vazia em metros quadrados
        - 'non_empty_area_hectares': área não vazia em hectares
        - 'non_empty_area_km2': área não vazia em quilômetros quadrados
        - 'total_area_hectares': área total da imagem em hectares
        - 'crs': CRS original da imagem
        - 'dimensions': tupla (width_pixels, height_pixels, num_bands)
    
    Examples:
    ---------
    >>> result = calculate_non_empty_area_hectares('minha_imagem.tif')
    >>> print(f"Área não vazia: {result['non_empty_area_hectares']:.2f} hectares")
    >>> print(f"Cobertura: {result['percent_non_empty']:.1f}%")
    """
    
    with rasterio.open(image_path) as src:
        # Informações básicas
        width = src.width
        height = src.height
        num_bands = src.count
        
        if verbose:
            print(f"Dimensões: {width} x {height} pixels ({num_bands} bandas)")
            print(f"CRS original: {src.crs}")
        
        # Obter o CRS
        crs = src.crs
        
        if crs is None:
            raise ValueError("⚠️  Imagem não possui CRS definido! Não é possível calcular a área.")
        
        # Calcular área de um pixel em metros quadrados
        crs_obj = CRS.from_string(str(crs))
        is_geographic = crs_obj.is_geographic
        is_projected = crs_obj.is_projected
        
        if verbose:
            print(f"\nTipo de CRS:")
            print(f"  - Geográfico (lat/lon): {is_geographic}")
            print(f"  - Projetado: {is_projected}")
        
        # Obter informações sobre as unidades
        if len(crs_obj.axis_info) >= 2:
            unit_x = crs_obj.axis_info[0].unit_name
        else:
            unit_x = crs_obj.axis_info[0].unit_name if crs_obj.axis_info else 'unknown'
        
        # Obter resolução do pixel
        pixel_width = abs(src.transform[0])
        pixel_height = abs(src.transform[4])
        
        if verbose:
            print(f"  - Unidade: {unit_x}")
            print(f"\nResolução do pixel: {pixel_width} x {pixel_height} ({unit_x})")
        
        # Calcular área de um pixel baseado no tipo de CRS
        if is_projected:
            conversion_factor = get_unit_to_meter_conversion(unit_x)
            
            if conversion_factor is not None:
                # Conversão direta
                if verbose:
                    print(f"\n✓ CRS projetado em {unit_x}. Fator de conversão: {conversion_factor}")
                
                pixel_area_m2 = (pixel_width * conversion_factor) * (pixel_height * conversion_factor)
                
            else:
                # Usar transformação para calcular área média de pixel
                if verbose:
                    print(f"\n⚠️  Unidade '{unit_x}' não reconhecida. Usando transformação...")
                
                bounds = src.bounds
                try:
                    transformer_to_geo = Transformer.from_crs(crs, 'EPSG:4326', always_xy=True)
                    center_lon, center_lat = transformer_to_geo.transform(
                        (bounds.left + bounds.right) / 2,
                        (bounds.bottom + bounds.top) / 2
                    )
                except:
                    raise ValueError("Não foi possível determinar coordenadas geográficas do raster.")
                
                utm_zone = int((center_lon + 180) / 6) + 1
                hemisphere = 'north' if center_lat >= 0 else 'south'
                
                if hemisphere == 'north':
                    dst_crs = CRS.from_string(f'EPSG:326{utm_zone:02d}')
                else:
                    dst_crs = CRS.from_string(f'EPSG:327{utm_zone:02d}')
                
                # Calcular área transformando um pixel de referência
                transformer = Transformer.from_crs(crs, dst_crs, always_xy=True)
                
                # Pixel no centro da imagem
                center_x = (bounds.left + bounds.right) / 2
                center_y = (bounds.bottom + bounds.top) / 2
                
                # Cantos do pixel central
                corners = [
                    (center_x, center_y),
                    (center_x + pixel_width, center_y),
                    (center_x + pixel_width, center_y - pixel_height),
                    (center_x, center_y - pixel_height)
                ]
                corners_transformed = [transformer.transform(x, y) for x, y in corners]
                
                # Área do pixel usando fórmula de Shoelace
                width_m = abs(corners_transformed[1][0] - corners_transformed[0][0])
                height_m = abs(corners_transformed[2][1] - corners_transformed[1][1])
                pixel_area_m2 = width_m * height_m
                
        elif is_geographic:
            # CRS geográfico - calcular área de pixel transformando para UTM
            if verbose:
                print(f"\n⚠️  CRS geográfico (lat/lon). Transformando para calcular área de pixel...")
            
            bounds = src.bounds
            center_lon = (bounds.left + bounds.right) / 2
            center_lat = (bounds.bottom + bounds.top) / 2
            
            utm_zone = int((center_lon + 180) / 6) + 1
            hemisphere = 'north' if center_lat >= 0 else 'south'
            
            if hemisphere == 'north':
                dst_crs = CRS.from_string(f'EPSG:326{utm_zone:02d}')
            else:
                dst_crs = CRS.from_string(f'EPSG:327{utm_zone:02d}')
            
            if verbose:
                print(f"  Zona UTM: {utm_zone}{hemisphere[0].upper()}")
            
            transformer = Transformer.from_crs(crs, dst_crs, always_xy=True)
            
            # Calcular área de um pixel de referência no centro
            corners = [
                (center_lon, center_lat),
                (center_lon + pixel_width, center_lat),
                (center_lon + pixel_width, center_lat - pixel_height),
                (center_lon, center_lat - pixel_height)
            ]
            corners_transformed = [transformer.transform(x, y) for x, y in corners]
            
            width_m = abs(corners_transformed[1][0] - corners_transformed[0][0])
            height_m = abs(corners_transformed[2][1] - corners_transformed[1][1])
            pixel_area_m2 = width_m * height_m
            
        else:
            raise ValueError("⚠️  Tipo de CRS não reconhecido!")
        
        if verbose:
            print(f"  Área de 1 pixel: {pixel_area_m2:.6f} m²")
        
        # Ler os dados da imagem e identificar pixels não vazios
        if verbose:
            print(f"\n📊 Analisando pixels não vazios...")
        
        if num_bands == 1:
            # Imagem em escala de cinza
            data = src.read(1)
            non_empty_mask = data != 0
            
        else:
            # Imagem multiband (RGB ou outras)
            # Um pixel é considerado vazio se TODOS os canais forem zero
            data = src.read()  # Shape: (bands, height, width)
            
            # Pixel é não vazio se ALGUM canal for diferente de zero
            non_empty_mask = np.any(data != 0, axis=0)
        
        # Contar pixels
        total_pixels = width * height
        non_empty_pixels = np.sum(non_empty_mask)
        empty_pixels = total_pixels - non_empty_pixels
        percent_non_empty = (non_empty_pixels / total_pixels) * 100
        
        # Calcular áreas
        non_empty_area_m2 = non_empty_pixels * pixel_area_m2
        non_empty_area_hectares = non_empty_area_m2 / 10000
        non_empty_area_km2 = non_empty_area_hectares / 100
        
        total_area_m2 = total_pixels * pixel_area_m2
        total_area_hectares = total_area_m2 / 10000
        
        if verbose:
            print(f"\n{'='*50}")
            print(f"RESULTADOS:")
            print(f"{'='*50}")
            print(f"Total de pixels: {total_pixels:,}")
            print(f"Pixels não vazios: {non_empty_pixels:,} ({percent_non_empty:.2f}%)")
            print(f"Pixels vazios: {empty_pixels:,} ({100-percent_non_empty:.2f}%)")
            print(f"\nÁrea por pixel: {pixel_area_m2:.6f} m²")
            print(f"\nÁrea total da imagem: {total_area_hectares:,.2f} hectares")
            print(f"Área não vazia: {non_empty_area_m2:,.2f} m²")
            print(f"Área não vazia: {non_empty_area_hectares:,.2f} hectares")
            print(f"Área não vazia: {non_empty_area_km2:,.2f} km²")
            print(f"{'='*50}")
        
        return {
            'total_pixels': int(total_pixels),
            'non_empty_pixels': int(non_empty_pixels),
            'empty_pixels': int(empty_pixels),
            'percent_non_empty': float(percent_non_empty),
            'pixel_area_m2': float(pixel_area_m2),
            'non_empty_area_m2': float(non_empty_area_m2),
            'non_empty_area_hectares': float(non_empty_area_hectares),
            'non_empty_area_km2': float(non_empty_area_km2),
            'total_area_hectares': float(total_area_hectares),
            'crs': str(crs),
            'dimensions': (width, height, num_bands)
        }


In [ ]:
# Exemplo de uso: calcular área não vazia
image_path = "/home/luizluz/Documentos/multi-task-fcn/amazon_input_data/orthoimage/NOV_2017_FINAL_004.tif"

result = calculate_non_empty_area_hectares(image_path)

print("\n\n📊 Resumo comparativo:")
print(f"   - Área total da imagem: {result['total_area_hectares']:.2f} hectares")
print(f"   - Área não vazia: {result['non_empty_area_hectares']:.2f} hectares ({result['percent_non_empty']:.1f}%)")
print(f"   - Área vazia (zeros): {result['total_area_hectares'] - result['non_empty_area_hectares']:.2f} hectares")


In [ ]:
# Exemplo de uso da função
image_path = "/home/luizluz/Documentos/multi-task-fcn/amazon_input_data/orthoimage/NOV_2017_FINAL_004.tif"

# Chamar a função com saída detalhada (verbose=True por padrão)
result = calculate_raster_area_hectares(image_path)

print("\n\n📊 Resumo dos resultados:")
print(f"   - Área: {result['area_hectares']:.2f} hectares")
print(f"   - Área: {result['area_km2']:.2f} km²")
print(f"   - Dimensões: {result['dimensions'][0]} x {result['dimensions'][1]} pixels")
